# LLM-OS: Fine-tune Gemma 4 E4B on ISA bytecode traces

This notebook fine-tunes **Gemma 4 E4B** (4B params) with **QLoRA** via [Unsloth](https://unsloth.ai) on execution traces generated by LLM-OS v3.

The fine-tuned model learns to emit ISA opcodes (call, halt, loop, think, etc.) from game state, replacing the large cloud model (Gemma 4 31B / Gemini Flash Lite) with a small local model that runs via llama.cpp or Ollama.

**Requirements:**
- Google Colab with T4 GPU (free tier works)
- A JSONL dataset generated by `v3/bin/llm-os.js --export-dataset --export-steps`

**Pipeline:**
```
v3 (generate traces) -> this notebook (fine-tune) -> v2 (run locally)
```

## 1. Install Unsloth

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

## 2. Load Gemma 4 E4B with 4-bit quantization

E4B fits on a free Colab T4 GPU (~10GB VRAM with QLoRA).
Unsloth recommends E4B QLoRA over E2B LoRA for better quality.

In [ ]:
from unsloth import FastModel
import torch

MAX_SEQ_LENGTH = 4096  # ISA traces can be long

model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-4-E4B-it",
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,   # QLoRA: 4-bit base + trainable LoRA adapters
    dtype=None,          # auto-detect
    full_finetuning=False,
)

print(f"Model loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 3. Configure LoRA adapters

We target all attention + MLP projection layers for maximum ISA learning.

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,                # LoRA rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

# Count trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 4. Upload and prepare the dataset

Upload the JSONL file generated by v3:
```bash
node bin/llm-os.js --cart cart/game/tetris \
  --task "Play tetris. Observe, think, rotate and move pieces, then drop." \
  --runs 100 --export-dataset dataset/tetris_steps.jsonl --export-steps
```

Each line is a JSON object with a `messages` array in OpenAI chat format.
Gemma uses `"model"` instead of `"assistant"` for the role, so we convert.

In [ ]:
from google.colab import files
import json

# Upload the JSONL dataset
print("Upload your JSONL dataset file:")
uploaded = files.upload()
DATASET_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATASET_FILE}")

In [ ]:
import json
from datasets import Dataset

# Load JSONL and convert roles for Gemma format
examples = []
with open(DATASET_FILE) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        ex = json.loads(line)
        messages = ex["messages"]

        # Convert "assistant" -> "model" for Gemma chat format
        converted = []
        for msg in messages:
            role = msg["role"]
            if role == "assistant":
                role = "model"
            converted.append({"role": role, "content": msg["content"]})

        examples.append({"messages": converted})

dataset = Dataset.from_list(examples)
print(f"Dataset: {len(dataset)} examples")
print(f"\nSample (first 3 messages of example 0):")
for msg in dataset[0]["messages"][:3]:
    content = msg["content"][:120] + "..." if len(msg["content"]) > 120 else msg["content"]
    print(f"  [{msg['role']}] {content}")

## 5. Tokenize with Gemma 4 chat template

We apply the tokenizer's chat template so the model sees properly formatted turns.

In [ ]:
def format_chat(example):
    """Apply Gemma 4 chat template to messages."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = dataset.map(format_chat)

# Show token lengths
lengths = [len(tokenizer.encode(ex["text"])) for ex in dataset]
print(f"Token lengths: min={min(lengths)}, max={max(lengths)}, avg={sum(lengths)/len(lengths):.0f}")
print(f"Examples > {MAX_SEQ_LENGTH} tokens: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}")

## 6. Train with SFTTrainer

Training config tuned for ISA trace learning on T4 GPU.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        output_dir="outputs_llmos",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        save_strategy="epoch",
        dataset_num_proc=1,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"\nDone! Loss: {stats.training_loss:.4f}")

## 7. Test the fine-tuned model

Quick inference test: give it a Tetris task and see if it emits valid ISA opcodes.

In [ ]:
from transformers import TextStreamer

# Test with a Tetris task
test_messages = [
    {
        "role": "user",
        "content": "Play tetris. Observe the board, think about piece placement, rotate and move pieces, then drop."
    }
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

print("Model output:")
print("=" * 60)
_ = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.3,
    top_p=0.95,
    top_k=64,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)
print("=" * 60)

## 8. Export to GGUF

Export the fine-tuned model as GGUF for local inference with llama.cpp or Ollama.
Q4_K_M is a good balance of size (~3-4GB) and quality.

In [ ]:
# Save LoRA adapter (small, needs base model)
model.save_pretrained("llmos_gemma4_e4b_lora")
tokenizer.save_pretrained("llmos_gemma4_e4b_lora")
print("LoRA adapter saved.")

# Export merged GGUF (standalone, for llama.cpp / Ollama)
model.save_pretrained_gguf(
    "llmos_gemma4_e4b_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF Q4_K_M exported.")

In [ ]:
# Download the GGUF file
import glob
gguf_files = glob.glob("llmos_gemma4_e4b_gguf/*.gguf")
if gguf_files:
    print(f"GGUF files: {gguf_files}")
    for f in gguf_files:
        size_mb = os.path.getsize(f) / 1e6
        print(f"  {f}: {size_mb:.0f} MB")
    files.download(gguf_files[0])
else:
    print("No GGUF files found. Check the export step above.")

## 9. (Optional) Push to Hugging Face Hub

Upload the LoRA adapter or GGUF to HF Hub for easy access.

In [ ]:
# Uncomment and set your HF details to push:
# HF_REPO = "your-username/llmos-gemma4-e4b-tetris"
# HF_TOKEN = "hf_..."  # from https://huggingface.co/settings/tokens

# # Push LoRA adapter
# model.push_to_hub(HF_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)

# # Push GGUF
# model.push_to_hub_gguf(
#     HF_REPO + "-gguf",
#     tokenizer,
#     quantization_method="q4_k_m",
#     token=HF_TOKEN,
# )

## Next steps

1. **Run locally**: Copy the GGUF file and run with llama.cpp or Ollama
2. **Integrate with v2**: Point v2's backend to the local model instead of OpenRouter
3. **Iterate**: Generate more traces with v3, re-train, improve

```bash
# Run with Ollama
ollama create llmos-tetris -f Modelfile
# where Modelfile points to the GGUF

# Run with llama.cpp
llama-server -m llmos_gemma4_e4b_gguf/*.gguf --port 8080
```